# 4-Class Eye Disease Classification - AlexNet (Kaggle)

Upload this notebook to Kaggle, **add the dataset**, enable **GPU**, then **Run All**.

**Dataset:** [gunavenkatdoddi/eye-diseases-classification](https://www.kaggle.com/datasets/gunavenkatdoddi/eye-diseases-classification)

**Model:** ImageNet-pretrained **AlexNet** with a 4-class head.

## Pipeline (19 sections - same layout as the main classwork notebook)

1. Auto-locate dataset under `/kaggle/input` (kagglehub fallback)
2. Stratified 70% / 15% / 15% split
3. Moderate fundus augmentations + light class-balanced sampling
4. Two-phase training: head warm-up -> **full fine-tune**
5. AMP + cosine LR + early stopping
6. Test-time augmentation (TTA)
7. Full metrics, confusion matrices, predictions


## 1 - Imports


In [ ]:
import os
import random
import time
import copy
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.models import alexnet, AlexNet_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from IPython.display import display

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Model:", 'AlexNet')


## 2 - Config + auto-find dataset

1. **Add data** (best): right panel → **+ Add data** → `eye-diseases-classification`
2. Or turn **Settings → Internet → ON** so `kagglehub` can fetch it automatically
3. Enable GPU: **Settings → Accelerator → GPU**


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

IMAGE_SIZE = 224          # AlexNet
BATCH_SIZE = 64           # head-phase batch
EPOCHS_HEAD = 5           # phase 1: classifier only
EPOCHS_FINETUNE = 20      # phase 2
LR_HEAD = 1e-3
LR_FINETUNE = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
PATIENCE = 6
NUM_WORKERS = 2
USE_AMP = True
MODEL_NAME = 'alexnet'
BATCH_SIZE_FINETUNE = 48  # phase-2
USE_FOCAL = False        # CE ~92%; focal dropped test to ~88%
USE_TTA = True
FOCAL_GAMMA = 1.5
GLAUCOMA_SAMPLE_BOOST = 1.25
GLAUCOMA_LOSS_BOOST = 1.25
NORMAL_LOSS_BOOST = 1.05

KAGGLE_DATASET = "gunavenkatdoddi/eye-diseases-classification"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# Loose name matching for class folders
CLASS_KEYWORDS = {
    "cataract": ("cataract",),
    "diabetic_retinopathy": ("diabetic_retinopathy", "diabetic-retinopathy", "retinopathy"),
    "glaucoma": ("glaucoma",),
    "normal": ("normal", "healthy"),
}

if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/alexnet_eye_4class_results")
else:
    OUTPUT_DIR = Path("./alexnet_eye_4class_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _subdir_names(path: Path):
    return [
        p for p in path.iterdir()
        if p.is_dir() and not p.name.startswith(".")
    ]


def _has_images(folder: Path, min_files: int = 1) -> bool:
    count = 0
    try:
        for p in folder.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                count += 1
                if count >= min_files:
                    return True
    except Exception:
        return False
    return False


def _match_classes(subdirs):
    """Return True if folders cover all 4 disease classes (fuzzy names)."""
    names = [p.name.lower().replace(" ", "_").replace("-", "_") for p in subdirs]
    matched = set()
    for class_key, keywords in CLASS_KEYWORDS.items():
        for name in names:
            if any(k.replace("-", "_") in name for k in keywords):
                matched.add(class_key)
                break
    return len(matched) == 4


def looks_like_dataset_root(path: Path) -> bool:
    if not path.is_dir():
        return False
    try:
        subdirs = _subdir_names(path)
    except Exception:
        return False
    if len(subdirs) < 4:
        return False
    if not _match_classes(subdirs):
        return False
    # At least one class folder should contain images
    return any(_has_images(d, min_files=1) for d in subdirs[:8])


def _print_input_tree(root: Path, max_depth: int = 3):
    print(f"\nScanning: {root}")
    if not root.exists():
        print("  (path does not exist)")
        return
    for dirpath, dirnames, filenames in os.walk(root):
        rel = Path(dirpath).relative_to(root)
        depth = 0 if str(rel) == "." else len(rel.parts)
        if depth > max_depth:
            dirnames[:] = []
            continue
        indent = "  " * depth
        label = "." if str(rel) == "." else rel.as_posix()
        n_img = sum(1 for f in filenames if Path(f).suffix.lower() in IMAGE_EXTS)
        print(f"{indent}{label}/  dirs={len(dirnames)} files={len(filenames)} images~={n_img}")


def _search_under(roots):
    preferred = []
    for root in roots:
        if not root:
            continue
        root = Path(root)
        if not root.exists():
            continue
        preferred.extend([
            root / "dataset",
            root,
            root / "eye-diseases-classification" / "dataset",
            root / "eye-diseases-classification",
            root / "dataset" / "dataset",
        ])

    seen = set()
    for p in preferred:
        key = str(p.resolve()) if p.exists() else str(p)
        if key in seen:
            continue
        seen.add(key)
        if looks_like_dataset_root(p):
            return p.resolve()

    # Broad walk (depth-limited)
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for dirpath, dirnames, _ in os.walk(root):
            path = Path(dirpath)
            depth = len(path.relative_to(root).parts) if path != root else 0
            if depth > 6:
                dirnames[:] = []
                continue
            if looks_like_dataset_root(path):
                return path.resolve()
    return None


def _download_with_kagglehub() -> Path:
    """Fallback: fetch dataset via kagglehub (needs Internet ON in Kaggle settings)."""
    try:
        import kagglehub
    except ImportError:
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub

    print(f"Dataset not in /kaggle/input — downloading via kagglehub: {KAGGLE_DATASET}")
    path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    print("kagglehub path:", path)
    found = _search_under([path, path.parent])
    if found is not None:
        return found
    if looks_like_dataset_root(path):
        return path.resolve()
    raise FileNotFoundError(f"kagglehub downloaded files, but class folders not found under: {path}")


def find_dataset_dir() -> Path:
    kaggle_input = Path("/kaggle/input")
    _print_input_tree(kaggle_input, max_depth=3)

    search_roots = []
    if kaggle_input.exists():
        search_roots.append(kaggle_input)
        # every attached dataset slug
        search_roots.extend(_subdir_names(kaggle_input))

    search_roots.extend([
        Path("/kaggle/input/eye-diseases-classification"),
        Path("/kaggle/input/eye-diseases-classification/dataset"),
        Path("./dataset"),
        Path("../input"),
        Path("/kaggle/working"),
        Path("."),
    ])

    found = _search_under(search_roots)
    if found is not None:
        return found

    # Auto fallback: kagglehub (works on Kaggle when Internet is enabled)
    try:
        return _download_with_kagglehub()
    except Exception as hub_err:
        raise FileNotFoundError(
            "Could not auto-find the eye disease dataset.\n"
            "Fix (pick one):\n"
            "  1) Kaggle right panel → + Add data → 'eye-diseases-classification'\n"
            "  2) Settings → Internet → ON  (allows kagglehub auto-download)\n"
            "Expected class folders: cataract / diabetic_retinopathy / glaucoma / normal\n"
            f"kagglehub error: {hub_err}"
        ) from hub_err


DATASET_DIR = find_dataset_dir()
print("\nDataset directory:", DATASET_DIR)
print("Class folders:", [p.name for p in _subdir_names(DATASET_DIR)])
print("Output directory :", OUTPUT_DIR)


## 3 - Reproducibility + device


In [ ]:
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Speed > strict determinism on Kaggle GPU
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 4 - Validate dataset + load ImageFolder


In [ ]:
class_folders = sorted(
    p.name for p in DATASET_DIR.iterdir()
    if p.is_dir() and not p.name.startswith(".")
)
print("Folders found:")
for folder in class_folders:
    print(" -", folder)

if len(class_folders) != 4:
    raise ValueError(f"Expected 4 class folders, found {len(class_folders)}: {class_folders}")

base_dataset = datasets.ImageFolder(root=str(DATASET_DIR))
CLASS_NAMES = base_dataset.classes
CLASS_TO_IDX = base_dataset.class_to_idx
NUM_CLASSES = len(CLASS_NAMES)

print("Class names:", CLASS_NAMES)
print("Class-to-index:", CLASS_TO_IDX)
print("Total images:", len(base_dataset))
assert NUM_CLASSES == 4


## 5 - Class distribution


In [ ]:
all_labels = np.array(base_dataset.targets)
all_indices = np.arange(len(base_dataset))
class_counts = Counter(all_labels)

class_distribution_df = pd.DataFrame([
    {
        "class_index": i,
        "class_name": name,
        "image_count": class_counts[i],
    }
    for i, name in enumerate(CLASS_NAMES)
])
display(class_distribution_df)
class_distribution_df.to_csv(OUTPUT_DIR / "original_class_distribution.csv", index=False)

plt.figure(figsize=(8, 5))
plt.bar(class_distribution_df["class_name"], class_distribution_df["image_count"], color="#2a6f97")
plt.title("Original Class Distribution")
plt.xlabel("Class")
plt.ylabel("Images")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "original_class_distribution.png", dpi=200, bbox_inches="tight")
plt.show()


## 6 - Stratified 70 / 15 / 15 split


In [ ]:
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=(VAL_RATIO + TEST_RATIO),
    random_state=SEED,
    stratify=all_labels,
)

temp_labels = all_labels[temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    random_state=SEED,
    stratify=temp_labels,
)

train_idx = np.array(train_idx)
val_idx = np.array(val_idx)
test_idx = np.array(test_idx)

assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
assert len(train_idx) + len(val_idx) + len(test_idx) == len(base_dataset)

print(f"Total : {len(base_dataset)}")
print(f"Train : {len(train_idx)} ({100 * len(train_idx) / len(base_dataset):.2f}%)")
print(f"Val   : {len(val_idx)} ({100 * len(val_idx) / len(base_dataset):.2f}%)")
print(f"Test  : {len(test_idx)} ({100 * len(test_idx) / len(base_dataset):.2f}%)")


## 7 - Save split assignments


In [ ]:
samples = base_dataset.samples
split_lookup = {int(i): "train" for i in train_idx}
split_lookup.update({int(i): "validation" for i in val_idx})
split_lookup.update({int(i): "test" for i in test_idx})

split_rows = []
for idx, (image_path, class_idx) in enumerate(samples):
    split_rows.append({
        "dataset_index": idx,
        "image_path": image_path,
        "image_name": Path(image_path).name,
        "class_index": class_idx,
        "class_name": CLASS_NAMES[class_idx],
        "split": split_lookup[idx],
    })

split_df = pd.DataFrame(split_rows)
display(split_df.head())
split_df.to_csv(OUTPUT_DIR / "fixed_70_15_15_split.csv", index=False)
for name in ["train", "validation", "test"]:
    split_df[split_df["split"] == name].to_csv(OUTPUT_DIR / f"{name}_split.csv", index=False)

split_distribution = pd.crosstab(split_df["class_name"], split_df["split"]).reindex(CLASS_NAMES)
display(split_distribution)
split_distribution.to_csv(OUTPUT_DIR / "split_class_distribution.csv")


## 8 - Transforms (moderate aug — matches the strong ~92% run)


In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.15),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.06), value="random"),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

print(train_transform)


## 9 - Datasets + loaders (glaucoma-boosted sampling)


In [ ]:
train_full = datasets.ImageFolder(root=str(DATASET_DIR), transform=train_transform)
val_full = datasets.ImageFolder(root=str(DATASET_DIR), transform=eval_transform)
test_full = datasets.ImageFolder(root=str(DATASET_DIR), transform=eval_transform)

assert train_full.class_to_idx == val_full.class_to_idx == test_full.class_to_idx

train_dataset = Subset(train_full, train_idx.tolist())
val_dataset = Subset(val_full, val_idx.tolist())
test_dataset = Subset(test_full, test_idx.tolist())

train_targets = all_labels[train_idx]
class_sample_count = np.bincount(train_targets, minlength=NUM_CLASSES).astype(np.float64)
class_weights_sampler = 1.0 / np.maximum(class_sample_count, 1.0)
for i, name in enumerate(CLASS_NAMES):
    if "glaucoma" in name.lower():
        class_weights_sampler[i] *= GLAUCOMA_SAMPLE_BOOST

sample_weights = class_weights_sampler[train_targets]
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

print("Sampler class weights:", {CLASS_NAMES[i]: float(class_weights_sampler[i]) for i in range(NUM_CLASSES)})

pin = torch.cuda.is_available()
persistent = NUM_WORKERS > 0
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=persistent,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=persistent,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=persistent,
)

images, labels = next(iter(train_loader))
print("Batch:", tuple(images.shape), labels[:8].tolist())
print("Train / Val / Test:", len(train_dataset), len(val_dataset), len(test_dataset))


## 10 - Sample training images


In [ ]:
def denormalize_image(tensor):
    mean = torch.tensor(imagenet_mean).view(3, 1, 1)
    std = torch.tensor(imagenet_std).view(3, 1, 1)
    return torch.clamp(tensor.cpu() * std + mean, 0, 1)

images, labels = next(iter(train_loader))
n = min(8, len(images))
plt.figure(figsize=(14, 7))
for i in range(n):
    plt.subplot(2, 4, i + 1)
    plt.imshow(denormalize_image(images[i]).permute(1, 2, 0).numpy())
    plt.title(CLASS_NAMES[labels[i].item()])
    plt.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sample_training_images.png", dpi=200, bbox_inches="tight")
plt.show()


## 11 - Build AlexNet

Phase-1 freezes the backbone and trains only the classifier.
Phase-2 fully fine-tunes all layers (default fine-tune).


In [ ]:
weights = AlexNet_Weights.DEFAULT
model = alexnet(weights=weights)
in_features = model.classifier[6].in_features
model.classifier[6] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)


def set_requires_grad(module, requires_grad: bool):
    for p in module.parameters():
        p.requires_grad = requires_grad


def freeze_backbone():
    set_requires_grad(model.features, False)
    set_requires_grad(model.classifier, True)


def unfreeze_for_finetune():
    for p in model.parameters():
        p.requires_grad = True


freeze_backbone()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total_params:,}")
print(f"Trainable (head) : {trainable_params:,}")


## 12 - Loss / metrics / train-eval (+ TTA)

Default is **CrossEntropy** (the ~92% setup). Focal is optional via `USE_FOCAL`.


In [ ]:
import torch.nn.functional as F
import warnings

warnings.filterwarnings("ignore", message=".*torch.cuda.amp.*")


class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, target):
        ce = F.cross_entropy(
            logits, target, weight=self.weight, reduction="none",
            label_smoothing=self.label_smoothing,
        )
        pt = torch.exp(-ce)
        return (((1.0 - pt) ** self.gamma) * ce).mean()


train_class_counts = np.bincount(all_labels[train_idx], minlength=NUM_CLASSES).astype(np.float64)
inv_freq = 1.0 / np.maximum(train_class_counts, 1.0)
class_loss_weights = inv_freq / inv_freq.mean()
for i, name in enumerate(CLASS_NAMES):
    lname = name.lower()
    if "glaucoma" in lname:
        class_loss_weights[i] *= GLAUCOMA_LOSS_BOOST
    elif "normal" in lname:
        class_loss_weights[i] *= NORMAL_LOSS_BOOST

class_weight_tensor = torch.tensor(class_loss_weights, dtype=torch.float32, device=device)
print("Loss class weights:", {CLASS_NAMES[i]: float(class_loss_weights[i]) for i in range(NUM_CLASSES)})

if USE_FOCAL:
    criterion = FocalLoss(
        weight=class_weight_tensor, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING
    )
else:
    criterion = nn.CrossEntropyLoss(
        weight=class_weight_tensor, label_smoothing=LABEL_SMOOTHING
    )

AMP_ENABLED = bool(USE_AMP and device.type == "cuda")
try:
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


def autocast_ctx():
    try:
        return torch.amp.autocast("cuda", enabled=AMP_ENABLED)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=AMP_ENABLED)


def compute_epoch_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {"accuracy": acc, "macro_precision": precision, "macro_recall": recall, "macro_f1": f1}


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_true, all_pred = [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast_ctx():
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        all_true.extend(labels.detach().cpu().numpy())
        all_pred.extend(preds.detach().cpu().numpy())
    metrics = compute_epoch_metrics(all_true, all_pred)
    metrics["loss"] = running_loss / len(loader.dataset)
    return metrics


def tta_logits(model, images):
    views = [
        images,
        torch.flip(images, dims=[3]),
        torch.flip(images, dims=[2]),
        torch.flip(images, dims=[2, 3]),
    ]
    return torch.stack([model(v) for v in views], dim=0).mean(dim=0)


@torch.no_grad()
def evaluate(model, loader, criterion, device, use_tta=False):
    model.eval()
    running_loss = 0.0
    all_true, all_pred, all_prob = [], [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast_ctx():
            outputs = tta_logits(model, images) if use_tta else model(images)
            loss = criterion(outputs, labels)
        probs = torch.softmax(outputs.float(), dim=1)
        preds = outputs.argmax(dim=1)
        running_loss += loss.item() * images.size(0)
        all_true.extend(labels.cpu().numpy())
        all_pred.extend(preds.cpu().numpy())
        all_prob.extend(probs.cpu().numpy())
    metrics = compute_epoch_metrics(all_true, all_pred)
    metrics["loss"] = running_loss / len(loader.dataset)
    return metrics, np.array(all_true), np.array(all_pred), np.array(all_prob)


print("Loss:", type(criterion).__name__, "| USE_FOCAL=", USE_FOCAL, "| TTA=", USE_TTA)
print("AMP enabled:", AMP_ENABLED)


## 13 - Two-phase training with early stopping


In [ ]:
BEST_MODEL_PATH = OUTPUT_DIR / "best_alexnet_eye_4class.pth"
history = []
best_val_macro_f1 = -np.inf
best_state = None
epochs_without_improvement = 0
global_epoch = 0

training_start = time.time()


def run_phase(phase_name, n_epochs, optimizer, scheduler=None):
    global best_val_macro_f1, best_state, epochs_without_improvement, global_epoch

    print("\n" + "=" * 72)
    print(f"PHASE: {phase_name} | epochs={n_epochs}")
    print("=" * 72)

    for _ in range(n_epochs):
        global_epoch += 1
        t0 = time.time()

        train_m = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_m, _, _, _ = evaluate(model, val_loader, criterion, device)

        if scheduler is not None:
            scheduler.step()

        row = {
            "epoch": global_epoch,
            "phase": phase_name,
            "train_loss": train_m["loss"],
            "train_accuracy": train_m["accuracy"],
            "train_macro_f1": train_m["macro_f1"],
            "val_loss": val_m["loss"],
            "val_accuracy": val_m["accuracy"],
            "val_macro_precision": val_m["macro_precision"],
            "val_macro_recall": val_m["macro_recall"],
            "val_macro_f1": val_m["macro_f1"],
            "lr": optimizer.param_groups[0]["lr"],
            "epoch_seconds": time.time() - t0,
        }
        history.append(row)

        print(
            f"Epoch {global_epoch:02d} [{phase_name}] | "
            f"TrLoss {train_m['loss']:.4f} Acc {train_m['accuracy']:.4f} F1 {train_m['macro_f1']:.4f} || "
            f"VaLoss {val_m['loss']:.4f} Acc {val_m['accuracy']:.4f} F1 {val_m['macro_f1']:.4f} | "
            f"{row['epoch_seconds']:.1f}s"
        )

        if val_m["macro_f1"] > best_val_macro_f1 + 1e-4:
            best_val_macro_f1 = val_m["macro_f1"]
            epochs_without_improvement = 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save(
                {
                    "model_state_dict": best_state,
                    "class_names": CLASS_NAMES,
                    "class_to_idx": CLASS_TO_IDX,
                    "best_val_macro_f1": best_val_macro_f1,
                    "seed": SEED,
                    "image_size": IMAGE_SIZE,
                    "backbone": "alexnet",
                },
                BEST_MODEL_PATH,
            )
            print("  -> Best model saved.")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print(f"\nEarly stopping after {PATIENCE} epochs without val macro-F1 gain.")
            return True
    return False


# ----- Phase 1: classifier head -----
freeze_backbone()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(EPOCHS_HEAD, 1))
stopped = run_phase("head", EPOCHS_HEAD, optimizer, scheduler)

# ----- Phase 2: partial fine-tune -----
if not stopped:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    unfreeze_for_finetune()
    trainable = [p for p in model.parameters() if p.requires_grad]
    print(f"Phase-2 trainable params: {sum(p.numel() for p in trainable):,}")

    ft_batch = int(globals().get("BATCH_SIZE_FINETUNE", max(8, BATCH_SIZE // 2)))
    print(f"Phase-2 batch size: {ft_batch}")
    train_loader = DataLoader(
        train_dataset, batch_size=ft_batch, sampler=sampler,
        num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=(NUM_WORKERS > 0),
    )
    val_loader = DataLoader(
        val_dataset, batch_size=ft_batch, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=(NUM_WORKERS > 0),
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(EPOCHS_FINETUNE, 1))
    epochs_without_improvement = 0

    try:
        run_phase("finetune", EPOCHS_FINETUNE, optimizer, scheduler)
    except torch.cuda.OutOfMemoryError:
        print("CUDA OOM — retrying with batch_size=8")
        torch.cuda.empty_cache()
        unfreeze_for_finetune()
        train_loader = DataLoader(
            train_dataset, batch_size=8, sampler=sampler,
            num_workers=0, pin_memory=pin, persistent_workers=False,
        )
        val_loader = DataLoader(
            val_dataset, batch_size=8, shuffle=False,
            num_workers=0, pin_memory=pin, persistent_workers=False,
        )
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(EPOCHS_FINETUNE, 1))
        epochs_without_improvement = 0
        run_phase("finetune", EPOCHS_FINETUNE, optimizer, scheduler)

total_training_seconds = time.time() - training_start
print("\nTraining finished.")
print(f"Best validation macro-F1: {best_val_macro_f1:.4f}")
print(f"Total training time: {total_training_seconds / 60:.2f} min")
print("Checkpoint:", BEST_MODEL_PATH)


## 14 - Training curves


In [ ]:
history_df = pd.DataFrame(history)
display(history_df)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)

curve_specs = [
    ("train_loss", "val_loss", "Loss", "loss_curve.png"),
    ("train_accuracy", "val_accuracy", "Accuracy", "accuracy_curve.png"),
    ("train_macro_f1", "val_macro_f1", "Macro-F1", "macro_f1_curve.png"),
]

for train_key, val_key, ylabel, fname in curve_specs:
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df[train_key], marker="o", label="Train")
    plt.plot(history_df["epoch"], history_df[val_key], marker="o", label="Validation")
    plt.title(f"Training and Validation {ylabel}")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / fname, dpi=200, bbox_inches="tight")
    plt.show()


## 15 - Load best checkpoint + test evaluation (with TTA)


In [ ]:
try:
    checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
except TypeError:
    checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
print("Loaded best checkpoint | val macro-F1:", checkpoint["best_val_macro_f1"])

# Plain eval (for reference) + TTA eval (used for final metrics)
test_metrics_plain, _, _, _ = evaluate(
    model, test_loader, criterion, device, use_tta=False
)
print("Test WITHOUT TTA:")
for k, v in test_metrics_plain.items():
    print(f"  {k}: {v:.4f}")

test_metrics_basic, y_true, y_pred, y_prob = evaluate(
    model, test_loader, criterion, device, use_tta=USE_TTA
)
print()
print(f"Test WITH TTA (USE_TTA={USE_TTA}):")
for k, v in test_metrics_basic.items():
    print(f"  {k}: {v:.4f}")


## 16 - Class-wise + overall metrics


In [ ]:
class_precision, class_recall, class_f1, class_support = precision_recall_fscore_support(
    y_true, y_pred, labels=np.arange(NUM_CLASSES), average=None, zero_division=0
)

classwise_results_df = pd.DataFrame({
    "class_index": np.arange(NUM_CLASSES),
    "class_name": CLASS_NAMES,
    "precision": class_precision,
    "recall": class_recall,
    "f1_score": class_f1,
    "support": class_support,
})
print("CLASS-WISE TEST RESULTS")
display(classwise_results_df)
classwise_results_df.to_csv(OUTPUT_DIR / "test_classwise_metrics.csv", index=False)

accuracy = accuracy_score(y_true, y_pred)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)
weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

macro_results_df = pd.DataFrame([{
    "test_loss": test_metrics_basic["loss"],
    "accuracy": accuracy,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1": macro_f1,
    "weighted_precision": weighted_precision,
    "weighted_recall": weighted_recall,
    "weighted_f1": weighted_f1,
}])
print("OVERALL / MACRO TEST RESULTS")
display(macro_results_df)
macro_results_df.to_csv(OUTPUT_DIR / "test_macro_overall_metrics.csv", index=False)

report_dict = classification_report(
    y_true, y_pred, labels=np.arange(NUM_CLASSES),
    target_names=CLASS_NAMES, output_dict=True, zero_division=0,
)
report_df = pd.DataFrame(report_dict).transpose()
display(report_df)
report_df.to_csv(OUTPUT_DIR / "test_classification_report.csv")


## 17 - Confusion matrices


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
cm_df = pd.DataFrame(
    cm,
    index=[f"True_{c}" for c in CLASS_NAMES],
    columns=[f"Pred_{c}" for c in CLASS_NAMES],
)
display(cm_df)
cm_df.to_csv(OUTPUT_DIR / "confusion_matrix_raw.csv")

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES).plot(
    ax=ax, cmap="Blues", values_format="d", colorbar=False
)
plt.title("Test Confusion Matrix")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix_raw.png", dpi=200, bbox_inches="tight")
plt.show()

cm_normalized = confusion_matrix(
    y_true, y_pred, labels=np.arange(NUM_CLASSES), normalize="true"
)
cm_normalized_df = pd.DataFrame(
    cm_normalized,
    index=[f"True_{c}" for c in CLASS_NAMES],
    columns=[f"Pred_{c}" for c in CLASS_NAMES],
)
display(cm_normalized_df)
cm_normalized_df.to_csv(OUTPUT_DIR / "confusion_matrix_normalized.csv")

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix=cm_normalized, display_labels=CLASS_NAMES).plot(
    ax=ax, cmap="Blues", values_format=".2f", colorbar=False
)
plt.title("Normalized Test Confusion Matrix")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix_normalized.png", dpi=200, bbox_inches="tight")
plt.show()


## 18 - Save predictions + show errors


In [ ]:
ordered_test_rows = split_df.set_index("dataset_index").loc[list(test_idx)].reset_index()
assert len(ordered_test_rows) == len(y_true)

prediction_df = pd.DataFrame({
    "dataset_index": ordered_test_rows["dataset_index"].values,
    "image_path": ordered_test_rows["image_path"].values,
    "image_name": ordered_test_rows["image_name"].values,
    "true_index": y_true,
    "pred_index": y_pred,
    "true_class": [CLASS_NAMES[i] for i in y_true],
    "pred_class": [CLASS_NAMES[i] for i in y_pred],
    "correct": (y_true == y_pred),
})
for class_idx, class_name in enumerate(CLASS_NAMES):
    safe = class_name.replace(" ", "_").replace("/", "_").replace("\\", "_")
    prediction_df[f"prob_{safe}"] = y_prob[:, class_idx]

display(prediction_df.head())
prediction_df.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

misclassified_df = prediction_df[prediction_df["correct"] == False].copy()
print("Misclassified test images:", len(misclassified_df))
if len(misclassified_df) > 0:
    sample_errors = misclassified_df.head(8)
    plt.figure(figsize=(14, 7))
    for plot_idx, (_, row) in enumerate(sample_errors.iterrows(), start=1):
        image = plt.imread(row["image_path"])
        plt.subplot(2, 4, plot_idx)
        plt.imshow(image, cmap="gray" if image.ndim == 2 else None)
        plt.title(f"True: {row['true_class']}\nPred: {row['pred_class']}", fontsize=10)
        plt.axis("off")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "sample_misclassified_images.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("No misclassified images.")


## 19 - Final summary


In [ ]:
print("=" * 72)
print("FINAL ALEXNET EYE-DISEASE CLASSIFICATION SUMMARY")
print("=" * 72)
print(f"Dataset             : {DATASET_DIR}")
print(f"Classes             : {CLASS_NAMES}")
print(f"Total images        : {len(base_dataset)}")
print(f"Train / Val / Test  : {len(train_idx)} / {len(val_idx)} / {len(test_idx)}")
print(f"Best Val Macro-F1   : {best_val_macro_f1:.4f}")
print("-" * 72)
print(f"Test Accuracy       : {accuracy:.4f}")
print(f"Test Macro Precision: {macro_precision:.4f}")
print(f"Test Macro Recall   : {macro_recall:.4f}")
print(f"Test Macro F1       : {macro_f1:.4f}")
print("-" * 72)
print(classwise_results_df[["class_name", "precision", "recall", "f1_score", "support"]].to_string(index=False))
print("\nOutputs:", OUTPUT_DIR)


# End

## Why 92% became 88%

Val stayed high (~0.93) while test dropped — classic sign the training recipe overfit the wrong objective:

1. **Focal loss (gamma=2) + heavy glaucoma weights** — chased hard examples, hurt overall test
2. **Harsh augmentation** — train looked harder than real test images
3. **Deeper unfreeze / smaller batches** — noisier fine-tune

## This restored recipe

- CrossEntropy (`USE_FOCAL=False`)
- Moderate aug
- `UNFREEZE_BLOCKS=4`, `LR_FINETUNE=3e-4`
- Light glaucoma boost only
- OOM-safe phase-2 + TTA kept
